In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd

from tqdm import tqdm
from pathlib import Path
from datetime import timedelta
from shapely import wkt
import ast
from shapely.geometry import Point, LineString

datasets = Path("/nas/cee-water/cjgleason/data")
era5_dir = datasets / "ERA5-Land/sub_basin_timeseries"
swot_lake_dir = datasets / 'hydrocron' / 'lake'

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

matchups = gpd.read_parquet(metadata_dir / "All_MERIT_matchups.parquet").set_index('comid')
matchups.index = matchups.index.astype(str)
 
# Safely convert stringified lists/dicts back to Python objects
def safe_literal_eval(df, col):
    df[col] = df[col].apply(lambda x: ast.literal_eval(x))
    return df

for col in ["s2m_values", "lake_reach_ids", "lake_pld_ids"]:
    matchups = safe_literal_eval(matchups, col)

In [2]:
from data import BasinDataLake

root_dir = save_dir / 'datalakes' / 'training'
store = BasinDataLake(root_dir)
# store.compact()

In [3]:
processed_basins = store.get_processing_status(source='glow-s')
to_process = matchups[~matchups.index.isin(processed_basins['subbasin'])]
to_process

,outlet,outlet_id,total_area,unitarea,reservoir,custom,reach_id,sword_area,sword_distance,lake_reach_ids,...,longitude,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider,hybas_area_diff,geometry
comid,,,,,,,,,,,,,,,,,,,,,
21000001,POINT (5.889166666666659 47.94916666666667),EAUF-V7200010,398.5,152.9,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,-0.034201,"MULTIPOLYGON (((5.93875 47.99125, 5.93708 47.9..."
21000012,POINT (5.889166666666659 47.94916666666667),EAUF-V7200010,324.8,194.6,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.053341,"MULTIPOLYGON (((5.79042 48.01125, 5.79042 48.0..."
21000019,POINT (5.684999999999993 47.53416666666667),EAUF-V7200010,5173.0,242.5,False,False,2.160280e+10,4530.071328,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,-0.007977,"MULTIPOLYGON (((5.76042 47.56125, 5.76042 47.5..."
21000021,POINT (5.76916666666666 47.58083333333334),EAUF-V7200010,4758.7,8.4,False,False,2.160280e+10,4278.612301,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.003021,"MULTIPOLYGON (((5.80625 47.60125, 5.80708 47.6..."
21000022,POINT (5.80416666666666 47.57),EAUF-V7200010,4506.6,68.9,False,False,2.160280e+10,4278.612301,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.009101,"MULTIPOLYGON (((5.85875 47.57458, 5.85875 47.5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-410401112134801,POINT (-112.1875 41.0625),USGS-410401112134801,9900.3,296.5,False,True,NaN,NaN,NaN,[],...,-112.230256,2003-10-01,2025-09-08,0.006230,45.873291,11.263905,7165.0,usgs,-0.002093,"MULTIPOLYGON (((-112.21458 40.96042, -112.2154..."
USGS-50035000,POINT (-66.4592 18.3217),USGS-50037000,345.7,345.7,False,True,NaN,NaN,NaN,[],...,-66.459568,1950-01-01,2025-09-08,0.240693,1857.585136,7.406389,25691.0,usgs,-0.091982,"MULTIPOLYGON (((-66.49208 18.29625, -66.49208 ..."
USGS-50037000,POINT (-66.5 18.3983),USGS-50037000,429.1,83.4,False,True,NaN,NaN,NaN,[],...,-66.496560,2019-06-13,2025-09-08,1.265763,911.802460,11.017469,2247.0,usgs,0.167805,"MULTIPOLYGON (((-66.50125 18.33125, -66.50125 ..."


In [3]:
glow_dir = datasets / "GLOW-S" / "daily_reach_aggregated"

glow_dfs = []
for region_file in glow_dir.glob('*_daily_median.parquet'):
    glow_dfs.append(pd.read_parquet(region_file))
                    
glow = pd.concat(glow_dfs)
glow

8it [00:14,  1.79s/it]


width
COMID    date                  
11000054 2017-04-06  189.730401
         2017-05-06  316.912003
         2017-05-16  211.071961
         2017-05-26  250.880601
         2017-06-19  253.637483
...                         ...
86007272 2022-09-19  102.147594
         2022-09-20  117.095609
         2022-09-24   89.471511
         2022-09-28  153.471128
         2022-10-03   76.117319

[81541676 rows x 1 columns]

In [4]:
# processed_basins = store.get_processing_status(source='glow-s')
# to_process = matchups[~matchups.index.isin(processed_basins['subbasin'])]
to_process = matchups

def get_glow_s(merit_reaches):
    if not merit_reaches:
        return None

    merit_reaches = [int(r) for r in merit_reaches]
    try:
        glow_ix = glow.loc[merit_reaches].groupby('date').median()
    except KeyError:
        return None
    
    glow_ix.index = pd.to_datetime(glow_ix.index).tz_localize('UTC')
    return glow_ix.rename(columns={'width':'glow_width'})
    

max_batch_size = 256
for basin_id, basin_group in tqdm(to_process.groupby('outlet_id')):
    subbasin_data = {}
    for comid, row in basin_group.iterrows():
        merit_reaches = row['s2m_values']
        subbasin_data[comid] = get_glow_s(merit_reaches)

        if len(subbasin_data)==max_batch_size:
            store.write_dynamic(basin_id, 'glow-s', subbasin_data, mode='append')
            subbasin_data = {}
            
    # Write any remaining data.
    if len(subbasin_data)>0:
        store.write_dynamic(basin_id, 'glow-s', subbasin_data, mode='append')

100%|██████████| 643/643 [1:13:40<00:00,  6.88s/it]  


In [ ]:
store.read_dynamic(basin_id, ['56115337'], ['glow-s'])